# 1. Long files

Turns the country questionnaires into long ("tidy") files, one folder per
chapter, and translates every Arabic one into English on the way.

```
DATA COLLECTOR\datacollector_received_quest\<Chapter>\*.xlsx
        -> COMPENDIUM ARAB SOCIETY - V2\Merged_long_files\<Chapter>_merged_long_files\
               <Chapter>_EN.xlsx   every questionnaire, in English
               <Chapter>_AR.xlsx   the Arabic questionnaires, as they arrived
```

One folder holds every questionnaire, in either language, with a subfolder per
chapter - Education, Health, Housing, Labor, Population, Poverty - and every
chapter folder is read. Nothing about the folder says which language a file is
in, so each file is checked by one simple rule (see `detect_language()`): **does
any of its column headers contain an Arabic letter?**

* an **Arabic** file is translated to English with `translation dict_V2.xlsx`;
* an **English** file is kept exactly as it is.

Everything is then appended into `<Chapter>_EN.xlsx`. The Arabic files are also
saved together, untranslated, as `<Chapter>_AR.xlsx` - notebook 2 calculates on
it, and notebook 5 builds the Arabic tabulations from it.

Anything the dictionary cannot translate passes through as it is and is listed
in `need manual intervention_<Chapter>.txt`, and in `GAPS` at the bottom.

Run this notebook first, then `Compendium_2_New_Indicators.ipynb`.

In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration - paths, chapters, the fuzzy-match cutoff, and the column
names the pipeline creates for itself.
"""
DATA_COLLECTOR_PATH = Path("C:/RAFFI/DSS-local") / "DATA COLLECTOR"
COMPENDIUM_PATH = Path("C:/RAFFI/DSS-local") / "COMPENDIUM ARAB SOCIETY - V2"

# Every questionnaire, Arabic or English, in one folder - a subfolder per
# chapter (Education, Health, Housing, Labor, Population, Poverty), and every
# one is read. Nothing about the folder says which language a file is in;
# detect_language() reads that off the file's own column headers.
QUESTIONNAIRES_PATH = DATA_COLLECTOR_PATH / "datacollector_received_quest"

TRANSLATION_DICT_PATH = COMPENDIUM_PATH / "translation dict_V2.xlsx"
# Confirmed readings for raw Value text - see clean_one_value() below.
VALUE_CORRECTIONS_PATH = DATA_COLLECTOR_PATH / "value corrections.xlsx"

# Leave CHAPTERS as None to process every chapter folder found on disk. Set an
# explicit list to restrict one run, e.g. CHAPTERS = ["Poverty"].
CHAPTERS = None

# Only used for a file whose column names are in neither script.
DEFAULT_LANGUAGE = "AR"

# A fuzzy match must score at least this well (0 to 1) to be used. Below it the
# value is left alone and a warning logged, rather than guessed at.
FUZZY_MATCH_CUTOFF = 0.6

# Column names the pipeline creates itself, written here in Arabic.
YEAR_COLUMN = "السنة"
VALUE_COLUMN = "العدد"
CHAPTER_COLUMN = "الفصل"

# Columns used to attach the right source/citation row to each data row:
# year, indicator, country.
MERGE_COLUMNS = ["السنة", "المؤشر", "الدولة"]

# Translated normally, but never fuzzy-matched: two citations differing by one
# digit score high enough to overwrite each other.
COLUMNS_NOT_FUZZY_MATCHED = ["المصدر"]


# ---------------------------------------------------------------------------
# Everything this run fixed, inferred, or added to the dictionary on its own
# goes to pipeline_changes_<Chapter>.txt; everything it left alone, and why,
# goes to need manual intervention_<Chapter>.txt - one pair of files per
# chapter, plus a _general pair for a dictionary-level finding with no
# chapter of its own. Each notebook owns a section and rewrites only its
# own, so either file always reflects the latest run of each step whatever
# order they ran in. See save_inconsistencies() below.


def changes_log_path(chapter):
    """Where one chapter's own auto-applied changes live - what this run
    corrected, inferred, or added to the dictionary on its own."""
    return COMPENDIUM_PATH / f"pipeline_changes_{chapter}.txt"


def manual_intervention_log_path(chapter):
    """Where one chapter's own open findings live - what this run left
    alone, and why, for a person to look at."""
    return COMPENDIUM_PATH / f"need manual intervention_{chapter}.txt"


GENERAL_CHANGES_LOG_PATH = COMPENDIUM_PATH / "pipeline_changes_general.txt"
GENERAL_MANUAL_LOG_PATH = COMPENDIUM_PATH / "need manual intervention_general.txt"

# A record's own `kind` decides which of the two files it lands in - fixed,
# inferred, or added to the dictionary goes to pipeline_changes_<Chapter>.txt;
# everything else (a gap, a skip, a collision) goes to
# need manual intervention_<Chapter>.txt. Anything not listed here defaults
# to manual intervention, on purpose - a new kind nobody has classified yet
# should surface for a person to see, not disappear into "already handled".
CHANGE_KINDS = {
    "column name corrected",
    "value corrected",
    "value needed correcting",
    "reused a close Arabic spelling",
    "dictionary entry added",
}


def _write_section(path, title, section, section_text):
    """Replace one named section in one file, in place, leaving every other
    section exactly as it was. The mechanic every chapter file and the
    general one share - split what's there into sections by name, replace
    this one, rebuild in section order so the file reads the same whatever
    order the notebooks last ran in."""
    header = [title, "=" * 78, ""]

    sections = {}
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body_text = "\n\n".join(sections[name] for name in sorted(sections))
    path.write_text("\n".join(header).rstrip("\n") + "\n\n" + body_text + "\n",
                    encoding="utf-8")


def save_inconsistencies(section, records, chapters=None):
    """Write this notebook's findings for this section, split into two files
    per chapter - pipeline_changes_<Chapter>.txt for everything this run
    fixed, inferred, or added to the dictionary on its own, and
    need manual intervention_<Chapter>.txt for everything it left alone and
    why. A finding with no chapter (a dictionary-level problem, not a
    source-data one) goes to the matching _general file instead.

    `chapters` is every chapter this call actually covers, independent of
    whether any of them have a finding - pass it explicitly so a clean
    chapter still gets both files correctly replaced with "Nothing found"
    rather than left showing whatever an earlier, unrelated run left there.
    """
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    marker = f"### {section} ###"

    WHERE = ["country", "indicator", "year", "sex", "age_group",
             "nationality", "area", "file", "sheet", "row"]

    def render(chapter_records):
        body = [marker, f"    last run {stamp}", ""]
        if not chapter_records:
            body += ["    Nothing found.", ""]
        else:
            frame = pd.DataFrame(chapter_records)
            for kind, group in frame.groupby("kind", sort=False):
                body.append(f"  {kind.upper()}  ({len(group)})")
                for _, row in group.iterrows():
                    def show(value):
                        if isinstance(value, float) and float(value).is_integer():
                            return str(int(value))
                        return str(value)
                    where = " · ".join(
                        show(row[f]) for f in WHERE
                        if f in row and pd.notna(row[f]) and str(row[f]) != "")
                    body.append(f"      {where}" if where else "      -")
                    body.append(f"          {row['detail']}")
                body.append("")
        return "\n".join(body)

    def split(chapter_records):
        changes = [r for r in chapter_records if r["kind"] in CHANGE_KINDS]
        manual = [r for r in chapter_records if r["kind"] not in CHANGE_KINDS]
        return changes, manual

    by_chapter = defaultdict(list)
    general = []
    for record in records:
        chapter = record.get("chapter")
        if chapter in (None, "", "-"):
            general.append(record)
        else:
            by_chapter[str(chapter)].append(record)

    covered = {str(c) for c in (chapters or [])} | set(by_chapter)

    paths = []
    for chapter in sorted(covered):
        changes, manual = split(by_chapter.get(chapter, []))
        _write_section(changes_log_path(chapter), f"PIPELINE CHANGES — {chapter}",
                       section, render(changes))
        _write_section(manual_intervention_log_path(chapter),
                       f"NEED MANUAL INTERVENTION — {chapter}", section, render(manual))
        paths += [changes_log_path(chapter), manual_intervention_log_path(chapter)]

    if general or not covered:
        changes, manual = split(general)
        _write_section(GENERAL_CHANGES_LOG_PATH, "PIPELINE CHANGES — general",
                       section, render(changes))
        _write_section(GENERAL_MANUAL_LOG_PATH, "NEED MANUAL INTERVENTION — general",
                       section, render(manual))
        paths += [GENERAL_CHANGES_LOG_PATH, GENERAL_MANUAL_LOG_PATH]

    return paths, len(records)

# Every long file lives under here, one folder per chapter:
# Merged_long_files\Population_merged_long_files\Population_EN.xlsx
LONG_FILES_ROOT = COMPENDIUM_PATH / "Merged_long_files"


def long_files_folder(chapter):
    """Population -> Merged_long_files/Population_merged_long_files."""
    return LONG_FILES_ROOT / f"{chapter}_merged_long_files"


def long_file_path(chapter, language):
    """<Chapter>_EN.xlsx or <Chapter>_AR.xlsx, in that chapter's own folder."""
    return long_files_folder(chapter) / f"{chapter}_{language}.xlsx"


## Load the translation dictionary

One dictionary, `translation dict_V2.xlsx`, read Arabic to English. It is used
twice here: as the vocabulary `correct_with_dictionary()` matches misspellings
against, and by `translate()` to turn every Arabic file into English.

In [ ]:
"""
CELL: Load translation dict_V2.xlsx - one dictionary, Arabic to English.
"""


def load_dictionary():
    """Reads translation dict_V2.xlsx into:

      DICTIONARY_AR_TO_EN  (column_map, value_map)
          column_map = {Arabic column name: English column name}
          value_map  = {Arabic column name: {Arabic value: English value}}

      ENGLISH_VOCABULARY  (column_names, values_by_column)
          the English column names and values the file knows about.

    Only the Arabic-to-English direction is built, because the questionnaires
    arrive in Arabic and English is what we translate into.

    ENGLISH_VOCABULARY is not a reverse dictionary - it maps nothing. It is just
    the list of correct English spellings, so an English questionnaire's
    misspelled labels can be fuzzy-matched against the right words too.
    """
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what misspellings are matched against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name_for(english_name, language):
    """Any dictionary column, spelled for the given language. column_name()
    below only knows the three names the pipeline invents for itself."""
    if language == "EN":
        return english_name
    english_to_arabic = {en: ar for ar, en in DICTIONARY_AR_TO_EN[0].items()}
    return english_to_arabic.get(english_name, english_name)


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


## `discover_chapters()`

Finds the chapter folders inside `datacollector_received_quest`.

In [ ]:
"""
CELL: discover_chapters() - find the chapter folders to read.
"""


def discover_chapters():
    """Chapter names taken from the folders on disk rather than a typed list.

    A chapter is any subfolder of datacollector_received_quest holding at
    least one .xlsx - so a new chapter needs no configuration, and an empty
    folder does not produce an empty output file.
    """
    if not QUESTIONNAIRES_PATH.exists():
        logger.warning(f"{QUESTIONNAIRES_PATH} does not exist - nothing to read")
        return []
    return sorted(
        child.name for child in QUESTIONNAIRES_PATH.iterdir()
        if child.is_dir() and any(
            f for f in child.glob("*.xlsx") if not f.name.startswith("~$")
        )
    )


def chapters_to_process():
    """CHAPTERS if it was set, otherwise whatever is on disk."""
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters discovered on disk: {found}")
    return found

## `extract_tables()`

Each sheet holds two tables marked by an `index` column: `index=1` is the data,
`index=2` is the source. The row reading `index` in column 0 carries the column
names for the table below it.


In [ ]:
"""
CELL: extract_tables() - split one raw sheet into its data table and source table.
"""


def extract_tables(raw_sheet):
    header_rows = raw_sheet.index[raw_sheet[0] == "index"].tolist()
    data_header_row, source_header_row = header_rows[0], header_rows[1]

    # .str.strip() matters here: a stray trailing space in a raw header (seen
    # in a couple of Morocco files) would otherwise silently break the merge
    # key match in reshape_and_merge() and cause a duplicate-column crash in
    # correct_with_dictionary().
    data_columns = raw_sheet.iloc[data_header_row].dropna().str.strip()
    data_table = raw_sheet[raw_sheet[0] == "1"][data_columns.index].copy()
    data_table.columns = data_columns.values
    data_table = data_table.drop(columns=["index"])

    source_columns = raw_sheet.iloc[source_header_row].dropna().str.strip()
    source_table = raw_sheet[raw_sheet[0] == "2"][source_columns.index].copy()
    source_table.columns = source_columns.values
    source_table = source_table.drop(columns=["index"])

    return data_table, source_table


## `detect_language()`

Decides whether a questionnaire file is Arabic or English, by one simple rule:
**the script its column headers are written in.** Arabic letters occupy their
own Unicode block (U+0600-U+06FF) and English text has nothing in it, so if at
least one of a file's headers contains an Arabic letter the file is Arabic;
only a file with no Arabic header at all is English. No dictionary is needed, so it still works on
a header the dictionary has never seen.

The headers are the ones on each sheet's `index` row - `المؤشر`, `الدولة`,
`الجنس`, ... in an Arabic file, `Indicator`, `Country`, `Sex`, ... in an
English one. Year columns are digits, in neither script, and are not counted.
Headers are a far better signal than cell values, which include numbers, codes
and citations that can be in either script.

The whole file is decided at once, from every data sheet's headers together,
and every sheet in it is processed in that language. Each sheet is also checked
on its own: one that disagrees with its file is reported rather than passed
over in silence.

In [ ]:
"""
CELL: detect_language() - is this file written in Arabic or English?
"""


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it."""
    return any("\u0600" <= character <= "\u06ff" for character in str(text))


def detect_language(*tables):
    """Returns "AR" or "EN" based on the script the column names are written in.

    Give it every data table in a file to decide the whole file, or a single
    sheet's table to check that one sheet. Arabic as soon as one column name
    holds an Arabic letter, English only when none does.

    Falls back to DEFAULT_LANGUAGE if there are no usable column names.
    """
    names = [str(c) for table in tables for c in table.columns if not str(c).isdigit()]
    if not names:
        return DEFAULT_LANGUAGE

    return "AR" if any(looks_arabic(name) for name in names) else "EN"

## `reshape_and_merge()`

Unpivots the data table's year columns (any column whose name is all
digits) into two columns (year, value), then merges in the matching row
from the source table so every data point carries its source.

Takes the sheet's `language` so the two columns it creates, and the columns
it merges on, are named in that same language.


In [ ]:
"""
CELL: reshape_and_merge() - wide-to-long reshape, then attach the source table.
"""


def reshape_and_merge(data_table, source_table, language):
    id_columns = [c for c in data_table.columns if not str(c).isdigit()]
    year_columns = [c for c in data_table.columns if str(c).isdigit()]

    long_table = data_table.melt(
        id_vars=id_columns,
        value_vars=year_columns,
        var_name=column_name(YEAR_COLUMN, language),
        value_name=column_name(VALUE_COLUMN, language),
    )

    # Merge on year/indicator/country spelled in the sheet's own language,
    # and only on the ones both tables actually have.
    merge_columns = [column_name(c, language) for c in MERGE_COLUMNS]
    merge_columns = [c for c in merge_columns if c in long_table.columns and c in source_table.columns]

    # Trim whitespace on BOTH sides of the merge keys first. This step runs
    # before correct_with_dictionary(), so a key with a stray trailing space
    # matches nothing in the source table and the row loses its citation
    # silently - the spelling is corrected a step later, too late to help. It is
    # not a rare case: of one Labor run's 358 fuzzy corrections, 331 were pure
    # whitespace, and one Algeria sheet lost ten citations to a single trailing
    # space on an indicator name.
    long_table = long_table.copy()
    source_table = source_table.copy()
    for column in merge_columns:
        long_table[column] = long_table[column].astype(str).str.replace("\xa0", " ").str.strip()
        source_table[column] = source_table[column].astype(str).str.replace("\xa0", " ").str.strip()

    return pd.merge(long_table, source_table, on=merge_columns, how="left")


## `correct_with_dictionary()`

Works in whichever language the sheet is written in - the `language` argument
picks which vocabulary to match against, so an English sheet is checked
against the English column names and values, an Arabic one against the Arabic.

For every column: if its name is already a known column name in that language,
leave it. Otherwise, find the known column name it's most similar to, and
rename it there - but only if that similarity score clears
`FUZZY_MATCH_CUTOFF`.

Then do the same thing for every value within each known column: if the
value is already known, leave it; otherwise replace it with the closest
known value, if close enough. Every actual change is printed.

Values in the **Source** column are never fuzzy-matched (see
`COLUMNS_NOT_FUZZY_MATCHED`) - two citations differing by a single digit
score high enough to overwrite each other, so the nearest entry would be a
wrong answer rather than a correction. They are still *translated* by exact
lookup. A misspelled Source *header* is still fixed, which is what folds
Iraq's `المصادر` back into the Source column.


In [ ]:
"""
CELL: correct_with_dictionary() - fix column names and cell values, in either language.
"""


# Every label the fuzzy matcher touches, matched or not - a value corrected is
# a spelling worth a record just as much as one nobody could place, so a
# reviewer can see everything the pipeline changed on its own, not only what it
# gave up on.
FINDINGS = []


def best_match(text, choices):
    """Compares text against every choice and returns (best_choice, score) -
    the one difflib considers most similar, and how similar (0 to 1)."""
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


def correct_with_dictionary(table, language, chapter, file_name, sheet_name):
    """Fixes misspelled column names and values by matching them against the
    dictionary's vocabulary for `language` ("AR" or "EN")."""
    known_columns, known_values_by_column = vocabulary(language)
    never_guessed = [column_name(c, language) for c in COLUMNS_NOT_FUZZY_MATCHED]

    table = table.copy()
    replacements = 0
    where = f"[{chapter}/{language}] {file_name} | {sheet_name}"

    # 1. Fix column names.
    for column in list(table.columns):
        if column in known_columns:
            continue  # already a known column name, nothing to fix

        match, score = best_match(column, known_columns.keys())
        if match is not None and score >= FUZZY_MATCH_CUTOFF:
            print(f"{where} | column: {column} -> {match} (score={score:.2f})")
            table = table.rename(columns={column: match})
            replacements += 1
            FINDINGS.append({
                "kind": "column name corrected", "chapter": chapter,
                "file": file_name, "sheet": sheet_name,
                "detail": f"{column!r} -> {match!r} (score {score:.2f})",
            })
        elif match is not None:
            FINDINGS.append({
                "kind": "column name not in the dictionary", "chapter": chapter,
                "file": file_name, "sheet": sheet_name,
                "detail": f"{column!r} - closest known name is {match!r} "
                          f"(score {score:.2f}), too far to use, so left unchanged",
            })
            logger.warning(
                f"{where} | column '{column}' has no good match "
                f"(closest is '{match}', score={score:.2f}) - left unchanged"
            )

    # 2. Fix cell values, one known column at a time.
    for column in table.columns:
        if column in never_guessed:
            continue  # free-text citations - translated, but never guessed at

        known_values = known_values_by_column.get(column)
        if not known_values:
            continue  # not a dictionary column, or it has no fixed vocabulary (e.g. Year, Value)

        for value in table[column].dropna().unique():
            if value in known_values:
                continue  # already a known value, nothing to fix

            match, score = best_match(value, known_values.keys())
            if match is not None and score >= FUZZY_MATCH_CUTOFF:
                print(f"{where} | {column}: {value} -> {match} (score={score:.2f})")
                table[column] = table[column].replace(value, match)
                replacements += 1
                FINDINGS.append({
                    "kind": "value corrected", "chapter": chapter,
                    "file": file_name, "sheet": sheet_name,
                    "detail": f"{column}: {value!r} -> {match!r} (score {score:.2f})",
                })
            elif match is not None:
                FINDINGS.append({
                    "kind": "value not in the dictionary", "chapter": chapter,
                    "file": file_name, "sheet": sheet_name,
                    "detail": f"{column}: {value!r} - closest known value is "
                              f"{match!r} (score {score:.2f}), too far to use",
                })
                logger.warning(
                    f"{where} | {column} value '{value}' has no good match "
                    f"(closest is '{match}', score={score:.2f}) - left unchanged"
                )

    return table, replacements


## `clean_values()`

Some Value cells cannot be used as a number as they stand, and each kind is
handled and then reported:

| the cell holds | what happens |
|---|---|
| `-`, `..`, `n/a` | blanked - it means "no data" |
| a cell of only spaces | blanked - it is not an empty cell, and survives every "is it blank" test downstream |
| `7845(الاعداد بالالف)` | **scaled**: the phrase means "figures in thousands", so `7845000` |
| `51.2+1.2` | read as the sum, `52.4` - taking `51.2` and dropping `+1.2` loses a figure the country did write |
| other wrapping text | dropped, keeping the bare number |

**A unit note describes the whole column it sits in, not the one cell it is
written on.** Morocco's 2024 population is filed as `1444`, `1745`, `1815`, …
with `الاعداد بالالف` typed into the first cell only. Scaling that one cell
would leave it a thousand times *larger* than its own neighbours — worse than
leaving it alone — so the multiplier is applied to every figure sharing that
indicator, country and year. The scope is the year, because that same file has
2010–2022 in whole units and only 2024 in thousands.

**Every change is reported**, including the block scaled by a note and how many
figures it moved. The pipeline cannot know for certain what an unrecognized
phrase meant, so it records what it did and leaves that judgement to whoever can
ask the country.


In [ ]:
"""
CELL: clean_values() - make the Value column usable, and say what was changed.
"""

# Cells that mean "no data" rather than a figure.
PLACEHOLDERS = {"-", "--", "---", "..", "...", "n/a", "na", "n.a.", "nil", "none"}

# A number possibly wrapped in other text: "7845(الاعداد بالالف)", "841 (الاعداد بالاف)".
NUMBER_IN_TEXT = re.compile(r"[-+]?\d[\d\s, ]*(?:\.\d+)?")

# A cell holding nothing but numbers added together - "51.2+1.2". Taking the
# first number and dropping "+1.2" is a guess that loses a figure the country
# did write, so it is read as the sum it looks like and reported as one.
SUM_EXPRESSION = re.compile(r"^\d[\d\s,]*(?:\.\d+)?(?:\s*\+\s*\d[\d\s,]*(?:\.\d+)?)+$")

# Arabic phrases naming the unit the figures are written in, and the multiplier
# that puts them back into whole units. "بالالف" and its common typo "بالاف"
# (missing one ل) both mean "in thousands"; the pattern covers both without
# listing them separately. Only a recognized phrase is trusted enough to scale a
# number - anything else is dropped as plain wrapping text.
UNIT_MULTIPLIERS = [
    (re.compile(r"بالمليون|بالملايين"), 1_000_000, "in millions"),
    (re.compile(r"بالال?ف"), 1_000, "in thousands"),
]


def load_value_corrections():
    """Raw Value text a person has confirmed a reading for, via the data
    quality notebook's review-and-apply loop.

    Keyed by (chapter, raw text) rather than by row or file: the same
    malformed cell text means the same thing everywhere it recurs within a
    chapter, so one confirmation answers every occurrence. Missing entirely
    until that notebook has been run and its review file applied at least
    once - clean_one_value() falls back to its own guesses either way.
    """
    if not VALUE_CORRECTIONS_PATH.exists():
        return {}
    table = pd.read_excel(VALUE_CORRECTIONS_PATH, engine="openpyxl")
    corrections = {}
    for chapter, raw, corrected in zip(
        table["chapter"], table["raw_value"], table["corrected_value"]
    ):
        if pd.isna(chapter) or pd.isna(raw) or pd.isna(corrected):
            continue
        corrections[(str(chapter).strip(), str(raw).strip())] = str(corrected).strip()
    return corrections


VALUE_CORRECTIONS = load_value_corrections()
if VALUE_CORRECTIONS:
    logger.info(f"Value corrections loaded: {len(VALUE_CORRECTIONS):,} confirmed reading(s)")


def unit_multiplier(text):
    """The multiplier a unit phrase asks for, as (factor, meaning).

    (1, None) when the text names no unit this knows.
    """
    for pattern, factor, meaning in UNIT_MULTIPLIERS:
        if pattern.search(text):
            return factor, meaning
    return 1, None


def scaled_text(number, factor):
    """`number` times `factor`, written without a pointless trailing '.0'."""
    value = float(number) * factor
    return str(int(value)) if value.is_integer() else str(round(value, 10))


def plain_number(value):
    """The value as a float when it is already a plain number, else None."""
    if pd.isna(value):
        return None
    text = re.sub(r"[\s,]", "", str(value).replace("\xa0", " "))
    if text == "":
        return None
    try:
        return float(text)
    except ValueError:
        return None


def clean_one_value(raw, chapter=None):
    """Return (cleaned, note, factor).

    `note` is None when nothing changed, otherwise a short phrase saying what
    happened - every change is reported, because a Value that needed correcting
    is a cell someone filled with something the pipeline cannot use as a number.
    `factor` is the unit multiplier this cell's own text asked for, which
    clean_values() then applies to the rest of the column it describes.
    """
    if pd.isna(raw):
        return raw, None, 1

    text = str(raw).replace("\xa0", " ").strip()
    if text == "":
        # A cell holding only spaces is not an empty one. It survives every
        # "is it blank" test downstream and reaches the charts as a value that
        # cannot be plotted, so it is blanked here and reported like any other
        # placeholder rather than passed through in silence.
        if str(raw) == "":
            return raw, None, 1
        return "", f"{str(raw)!r} holds only whitespace - means no data, blanked", 1

    # Already a plain number.
    plain = re.sub(r"[\s,]", "", text)
    try:
        float(plain)
        return raw, None, 1
    except ValueError:
        pass

    # A person has already looked at exactly this text, in the data quality
    # notebook's review file, and typed the answer - that answer wins over
    # every guess below it, because it is not a guess.
    confirmed = VALUE_CORRECTIONS.get((chapter, text))
    if confirmed is not None:
        return confirmed, f"{text!r} -> {confirmed} (confirmed reading, from value corrections.xlsx)", 1

    if text.lower() in PLACEHOLDERS:
        return "", f"{text!r} means no data - blanked", 1

    if SUM_EXPRESSION.match(text):
        parts = [float(re.sub(r"[\s,]", "", part)) for part in text.split("+")]
        total = scaled_text(sum(parts), 1)
        return total, (f"{text!r} -> {total} (read as a sum: "
                       f"{' + '.join(f'{part:g}' for part in parts)})"), 1

    match = NUMBER_IN_TEXT.search(text)
    if match:
        number = re.sub(r"[\s, ]", "", match.group(0))
        try:
            float(number)
        except ValueError:
            return raw, f"{text!r} is not a number and was left as it is", 1
        removed = text.replace(match.group(0), "").strip()

        factor, meaning = unit_multiplier(removed) if removed else (1, None)
        if factor != 1:
            value = scaled_text(number, factor)
            return value, (f"{text!r} -> {value} (dropped {removed!r}, meaning {meaning} "
                           f"- multiplied by {factor:,})"), factor

        return number, (f"{text!r} -> {number} (dropped {removed!r})"
                        if removed else f"{text!r} -> {number}"), 1

    return raw, f"{text!r} is not a number and was left as it is", 1


def clean_values(table, chapter, language, file_name=None):
    """Clean the Value column of one table, collecting a record per change.

    **A unit note describes the whole column it sits in, not the one cell it is
    written on.** Morocco's 2024 population is filed as 1444, 1745, 1815, ...
    with "الاعداد بالالف" ("figures in thousands") typed into the first cell
    only. Scaling that cell alone would leave it a thousand times *larger* than
    its own neighbours - worse than leaving it alone - so the multiplier is
    applied to every figure sharing that indicator, country and year, and the
    whole block is reported.

    The scope is the year: that same file has 2010-2022 in whole units and only
    2024 in thousands, so the note travels across the breakdowns of its own year
    and no further.

    Every change is reported either way - the source still needs correcting, and
    only the country that wrote it can say what was meant.
    """
    value_column = column_name(VALUE_COLUMN, language)
    if value_column not in table.columns:
        return table, []

    indicator_column = column_name_for("Indicator", language)
    country_column = column_name_for("Country", language)
    year_column = column_name(YEAR_COLUMN, language)

    table = table.copy()
    records = []

    def locate(position):
        """The country/indicator/year that pin one row down in the report."""
        where = {}
        for field, column in [("country", country_column),
                              ("indicator", indicator_column),
                              ("year", year_column)]:
            if column in table.columns:
                where[field] = table.at[position, column]
        return where

    # What a unit note describes: one indicator, one country, one year.
    block_columns = [c for c in (indicator_column, country_column, year_column)
                     if c in table.columns]
    blocks = defaultdict(list)
    for position in table.index:
        blocks[tuple(str(table.at[position, c]) for c in block_columns)].append(position)

    cleaned = {}
    scaled_blocks = {}
    # Every cell whose own text carried a unit note. A block can hold more than
    # one - Morocco 2024 has two - and each has already been scaled by its own
    # text, so the block pass below must skip all of them and not just the one
    # it happens to be triggered by.
    self_scaled = set()
    for position, raw in table[value_column].items():
        value, note, factor = clean_one_value(raw, chapter)
        cleaned[position] = value
        if note is not None:
            records.append({"kind": "value needed correcting", "chapter": chapter,
                            "detail": note, "file": file_name, **locate(position)})
        if factor != 1:
            self_scaled.add(position)
            key = tuple(str(table.at[position, c]) for c in block_columns)
            scaled_blocks.setdefault(key, (factor, position))

    for key, (factor, source_position) in scaled_blocks.items():
        changed = 0
        for position in blocks[key]:
            if position in self_scaled:
                continue          # its own text already scaled it, and was reported
            number = plain_number(cleaned[position])
            if number is None:
                continue          # blank, or something that is not a figure
            cleaned[position] = scaled_text(number, factor)
            changed += 1
        if changed:
            records.append({
                "kind": "value needed correcting", "chapter": chapter, "file": file_name,
                **locate(source_position),
                "detail": (f"the unit note on one cell describes the whole column: "
                           f"{changed} further figure(s) for this indicator, country and "
                           f"year were multiplied by {factor:,} to match it"),
            })

    table[value_column] = [cleaned[position] for position in table.index]
    return table, records


## `translate()`

An Arabic file's rows are translated to English here, once they have been
corrected and cleaned in Arabic: every Arabic value is swapped for its English
equivalent, then the column is renamed. An English file never goes through
this - it is already English.

Anything the dictionary has no entry for is left exactly as it is, and
`find_untranslated()` lists it - in `need manual intervention_<Chapter>.txt`
with the country, indicator and year it appears in, and in `GAPS`. Closing
those is the "Filling dictionary gaps" loop in `../CLAUDE.md`: fill in `val_en`
by position, call `update_dictionary(filled, chapter="<Chapter>")`, and re-run
this notebook.

Source citations are translated by exact lookup only, never fuzzy-matched -
see `COLUMNS_NOT_FUZZY_MATCHED`.

In [ ]:
"""
CELL: translate(), find_untranslated(), update_dictionary() - Arabic to
English, what the dictionary could not translate, and closing those gaps.
"""


def translate(table):
    """Swaps every Arabic value for its English equivalent, then renames the
    column to its English name.

    Values are replaced first and the column renamed second, because the value
    lookup is keyed by the column's ORIGINAL Arabic name - renaming first would
    lose it. Anything the dictionary has no entry for is left exactly as it is,
    and is picked up afterwards by find_untranslated().
    """
    column_map, value_map = DICTIONARY_AR_TO_EN

    table = table.copy()
    for column in list(table.columns):
        if column in value_map:
            table[column] = table[column].replace(value_map[column])
        if column in column_map:
            table = table.rename(columns={column: column_map[column]})
    return table


def find_untranslated(arabic_table, translated_table):
    """Values that came out of translate() unchanged and are still Arabic, in
    the shape update_dictionary() takes - val_en left blank to be filled in.

    A value with no dictionary entry is copied through untouched, so comparing
    the table before against after finds them exactly. Values already in
    Latin script are not gaps - plenty of Arabic questionnaires cite their
    source in English ("MICS 2022", a URL) and those are correct as they stand.
    """
    column_map, _ = DICTIONARY_AR_TO_EN

    def present(english_name):
        """The Arabic spelling of a column that this table actually uses."""
        return next((arabic for arabic, english in column_map.items()
                     if english == english_name and arabic in arabic_table.columns), None)

    context_columns = {field: present(name) for field, name in
                       [("country", "Country"), ("indicator", "Indicator"), ("year", "Year")]}

    gaps = []
    for arabic_column, english_column in zip(arabic_table.columns, translated_table.columns):
        before = arabic_table[arabic_column]
        after = translated_table[english_column]
        pairs = pd.DataFrame({"val_ar": before, "val_en": after}).dropna()

        for (arabic_value, english_value), count in pairs.groupby(["val_ar", "val_en"]).size().items():
            if str(arabic_value).strip() != str(english_value).strip():
                continue                      # translated fine
            if not looks_arabic(arabic_value):
                continue                      # already English
            # Where does it appear? One example row is enough to find it.
            example = arabic_table[arabic_table[arabic_column] == arabic_value].head(1)
            context = {field: example.iloc[0][column]
                       for field, column in context_columns.items()
                       if column is not None and len(example)}
            gaps.append({
                "col_ar": arabic_column, "col_en": english_column,
                "val_ar": arabic_value, "val_en": None, "rows": count, **context,
            })
    return gaps


def gap_records(gaps):
    """The dictionary gaps as records for the shared log, one per distinct
    value per chapter. Each carries the country, indicator and year of an
    example row, so a missing term can be traced back to the questionnaire it
    came from rather than only being named."""
    records, seen = [], set()
    for gap in gaps:
        key = (gap["chapter"], gap["col_ar"], gap["val_ar"])
        if key in seen:
            continue
        seen.add(key)
        record = {"kind": "no English translation", "chapter": gap["chapter"],
                  "detail": f"{gap['col_en']}: {gap['val_ar']!r} appears in "
                            f"{gap['rows']} row(s) and passes through untranslated"}
        for field in ("country", "indicator", "year"):
            if gap.get(field) is not None:
                record[field] = gap[field]
        records.append(record)
    return records


def dictionary_key(col_ar, val_ar):
    """The (column, value) pair that makes a dictionary row unique - with NaN
    normalized to None first. A float NaN never equals another float NaN
    (nan != nan is True), so leaving it as-is would make a column-defining
    row (val_ar genuinely blank, by design - see update_dictionary() below)
    look "new" every time, defeating the whole point of checking for it:
    called twice, it would add itself twice.
    """
    return (col_ar, val_ar if pd.notna(val_ar) else None)


DICTIONARY_LOG = []  # every row any update_dictionary() call has added this run


def update_dictionary(filled, chapter=None, backup=True):
    """Appends reviewed translations to translation dict_V2.xlsx.

    `filled` is GAPS with val_en filled in (a DataFrame, or a path to a saved
    file). An Arabic value the dictionary already has is left alone - so
    running this twice changes nothing the second time. A timestamped backup
    is written first, because this edits the project's source of truth.

    A row may teach a COLUMN name instead of a value translation - val_ar and
    val_en both genuinely blank, because a "column not in the dictionary" fix
    has no value to pair it with. Those are kept; only a row that is neither a
    complete value pair nor a column-only row is skipped.

    `chapter` names the run this call belongs to, so every row actually added
    is logged to that chapter's pipeline_changes_<Chapter>.txt. Left as None
    for a dictionary-wide call with no one chapter behind it, which logs to
    the _general file instead. The whole accumulated DICTIONARY_LOG is passed
    every time, so a second call in the same run never loses the first call's
    rows - a section is always rewritten in full.

    Re-run this notebook afterwards: the dictionary is read once, when the
    notebook starts.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    candidates = filled[needed].copy()
    is_column_row = candidates["val_ar"].isna() & candidates["val_en"].isna()
    column_rows = candidates[is_column_row
                             & candidates["col_ar"].notna() & candidates["col_en"].notna()]

    value_rows = candidates[~is_column_row].dropna()
    value_rows = value_rows[(value_rows["val_ar"].astype(str).str.strip() != "")
                            & (value_rows["val_en"].astype(str).str.strip() != "")]

    new_rows = pd.concat([column_rows, value_rows], ignore_index=True)
    if new_rows.empty:
        logger.warning("No completed rows to add - is val_en filled in?")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already_there = {dictionary_key(c, v)
                     for c, v in zip(dictionary["col_ar"], dictionary["val_ar"])}
    to_add = new_rows[~new_rows.apply(
        lambda r: dictionary_key(r["col_ar"], r["val_ar"]) in already_there, axis=1)]
    to_add = to_add.drop_duplicates(subset=["col_ar", "val_ar"])
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    # Mark what the pipeline added, so a row typed by hand and a row translated
    # by Claude Code can be told apart later. Originals keep a blank status.
    to_add = to_add.reindex(columns=dictionary.columns)
    if "status" in to_add.columns:
        to_add["status"] = "updated"

    updated = pd.concat([dictionary, to_add], ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
                f"({len(dictionary):,} -> {len(updated):,}). Re-run to use them.")

    DICTIONARY_LOG.extend({
        "kind": "dictionary entry added", "chapter": chapter or "-",
        "detail": (f"column name added: {row['col_ar']!r} -> {row['col_en']!r}"
                  if pd.isna(row["val_ar"]) else
                  f"{row['col_en']}: {row['val_ar']!r} -> {row['val_en']!r}"),
    } for _, row in to_add.iterrows())
    save_inconsistencies("1b. DICTIONARY", DICTIONARY_LOG,
                         chapters=[chapter] if chapter else None)

    return updated

## Error tracking

`log_failure()` is the one place that logs a failure and remembers it, so
the final cell can print one consolidated summary grouped by chapter.

In [ ]:
"""
CELL: Error tracking - log a failure and remember it for the run summary.
"""
FAILURES = defaultdict(list)  # chapter -> list of {file, sheet, step, error}

# Plain-language guess at what's wrong, keyed by which step failed.
LIKELY_CAUSES = {
    "read": "file may be corrupted, password-protected, or not a valid .xlsx",
    "extract": "sheet may be missing an 'index' column, or index values are not 1/2 as expected",
    "detect language": "sheet's column names match neither the Arabic nor the English dictionary",
    "reshape_and_merge": "data table may be missing year columns, or the merge columns don't match the source table",
    "dictionary correction": "column names or values may be malformed and unable to be matched",
    "clean values": "a Value cell holds something that is neither a number nor a known placeholder",
    "translation": "a column or value may not have a corresponding entry in the other language",
}


def log_failure(chapter, file_name, sheet_name, step, error):
    likely_cause = LIKELY_CAUSES.get(step, "unexpected error, inspect the sheet manually")
    logger.error(
        f"[ERROR] Chapter={chapter} | File={file_name} | Sheet={sheet_name} | "
        f"Step={step} | {type(error).__name__}: {error} | Likely cause: {likely_cause}"
    )
    FAILURES[chapter].append({
        "file": file_name,
        "sheet": sheet_name,
        "step": step,
        "error": f"{type(error).__name__}: {error}",
    })


## `process_chapter()`

Every step, for every sheet, for every file in one chapter folder. Each file's
language is decided first, from all of its sheets' headers together. Then each
sheet runs inside one try/except with a `step` variable tracking the stage, so
a failure is logged against the right step without needing a try/except per
step.

At the end the Arabic sheets are saved together as `<Chapter>_AR.xlsx` and
translated, and everything - translated Arabic and English as it came - is
appended into `<Chapter>_EN.xlsx`, both in the chapter's own
`Merged_long_files\<Chapter>_merged_long_files\` folder.

In [ ]:
"""
CELL: process_chapter() - every step for one chapter folder.
"""

# Every Arabic value translate() found no dictionary entry for, this run.
UNTRANSLATED = []


def read_sheets(chapter, file_path):
    """Every sheet of one questionnaire file split into its data and source
    tables, as (sheet_name, data_table, source_table).

    A sheet that cannot be read or split - a cover tab has no index rows at
    all - is logged and skipped, so one bad sheet never costs the rest of the
    file.
    """
    xls = pd.ExcelFile(file_path, engine="openpyxl")
    sheets = []
    for sheet_name in xls.sheet_names:
        step = "read"
        try:
            raw_sheet = pd.read_excel(xls, sheet_name=sheet_name, header=None, dtype=str)

            step = "extract"
            data_table, source_table = extract_tables(raw_sheet)
        except Exception as error:
            log_failure(chapter, file_path.name, sheet_name, step, error)
            continue
        sheets.append((sheet_name, data_table, source_table))
    return sheets


def process_chapter(chapter):
    """Builds one chapter's long files, and returns how many rows each got as
    {"EN": rows, "AR": rows} - empty if there was nothing to read, and with
    no "AR" when the chapter had no Arabic file."""
    folder = QUESTIONNAIRES_PATH / chapter
    if not folder.exists():
        logger.info(f"  {chapter}: no folder, skipping")
        return {}

    files = sorted(f for f in folder.glob("*.xlsx") if not f.name.startswith("~$"))
    if not files:
        logger.info(f"  {chapter}: folder is empty, skipping")
        return {}

    logger.info(f"  {chapter}: {len(files)} file(s)")

    tables = {"AR": [], "EN": []}
    sheets_done = 0
    replacements_made = 0

    for file_path in files:
        file_name = file_path.name
        try:
            sheets = read_sheets(chapter, file_path)
        except Exception as error:
            log_failure(chapter, file_name, "-", "read", error)
            continue
        if not sheets:
            continue

        # The file decides, from every data sheet's headers together.
        language = detect_language(*(data_table for _, data_table, _ in sheets))
        logger.info(f"    {file_name}: "
                    + ("Arabic - translated to English" if language == "AR"
                       else "English - kept as it is"))

        for sheet_name, data_table, source_table in sheets:
            step = "detect language"
            try:
                # A check only - the file decides. A sheet whose own headers
                # disagree is still read in its file's language, so its
                # labels will not match the vocabulary they are checked
                # against; saying so here beats a silent mess of unmatched rows.
                sheet_language = detect_language(data_table)
                if sheet_language != language:
                    FINDINGS.append({
                        "kind": "sheet in a different language from its file",
                        "chapter": chapter, "file": file_name, "sheet": sheet_name,
                        "detail": f"this sheet's headers look {sheet_language}, the "
                                  f"rest of the file's look {language} - read as {language}",
                    })
                    logger.warning(
                        f"    {file_name} | {sheet_name}: looks {sheet_language} but the "
                        f"file is {language} - processing it as {language}"
                    )

                chapter_in_language = chapter if language == "EN" else CHAPTER_TO_ARABIC[chapter]
                data_table[column_name(CHAPTER_COLUMN, language)] = chapter_in_language

                step = "reshape_and_merge"
                merged_table = reshape_and_merge(data_table, source_table, language)

                step = "dictionary correction"
                corrected_table, n = correct_with_dictionary(
                    merged_table, language, chapter, file_name, sheet_name
                )
                replacements_made += n

                step = "clean values"
                corrected_table, value_notes = clean_values(
                    corrected_table, chapter, language, file_name)
                FINDINGS.extend(value_notes)

            except Exception as error:
                log_failure(chapter, file_name, sheet_name, step, error)
                continue

            tables[language].append(corrected_table)
            sheets_done += 1

    if not sheets_done:
        logger.warning(f"  {chapter}: nothing extracted, no file written")
        return {}

    long_files_folder(chapter).mkdir(parents=True, exist_ok=True)
    written = {}
    english_parts = []

    arabic_path = long_file_path(chapter, "AR")
    if tables["AR"]:
        arabic = pd.concat(tables["AR"], ignore_index=True)
        arabic.to_excel(arabic_path, index=False, engine="openpyxl")
        written["AR"] = len(arabic)

        try:
            translated = translate(arabic)
            UNTRANSLATED.extend({"chapter": chapter, **gap}
                                for gap in find_untranslated(arabic, translated))
            english_parts.append(translated)
        except Exception as error:
            log_failure(chapter, arabic_path.name, "-", "translation", error)
    elif arabic_path.exists():
        logger.warning(f"  {chapter}: no Arabic file this run, but an older "
                       f"{arabic_path.name} is still on disk - notebook 2 and the "
                       f"Arabic tabulations will keep reading it until it is removed")

    # Concatenate, not merge side-by-side: both parts are the same long shape,
    # so this stacks the English files' rows under the translated ones.
    english_parts += tables["EN"]
    if not english_parts:
        return written
    english = pd.concat(english_parts, ignore_index=True)
    english_path = long_file_path(chapter, "EN")
    english.to_excel(english_path, index=False, engine="openpyxl")
    written["EN"] = len(english)

    logger.info(
        f"  {chapter}: {sheets_done} sheet(s), {replacements_made} correction(s); "
        f"{len(tables['AR'])} Arabic sheet(s) translated, {len(tables['EN'])} English "
        f"sheet(s) kept -> {english_path.parent.name}\\{english_path.name} "
        f"({len(english):,} rows)"
    )
    return written

## Run - build every long file

Walks every chapter folder in `datacollector_received_quest` in turn. A chapter
with no folder or no files is skipped and noted.

In [ ]:
"""
CELL: Main run - build the long files for every chapter folder.
"""
FAILURES.clear()
FINDINGS.clear()
UNTRANSLATED.clear()

chapters = chapters_to_process()
print(f"Reading questionnaires from: {QUESTIONNAIRES_PATH}")
print(f"Writing long files to:       {LONG_FILES_ROOT}\\<Chapter>_merged_long_files\n")

written = {}
for step_number, chapter in enumerate(chapters, start=1):
    bar = "#" * step_number + "-" * (len(chapters) - step_number)
    print(f"[{bar}] {step_number}/{len(chapters)}  {chapter}")
    rows = process_chapter(chapter)
    if rows:
        written[chapter] = rows

print("\n" + "=" * 70)
print("LONG FILES WRITTEN")
print("=" * 70)
if not written:
    print("None - check the questionnaire folder.")
else:
    for chapter, rows in sorted(written.items()):
        for language in ("EN", "AR"):
            if language in rows:
                path = long_file_path(chapter, language)
                print(f"  {path.parent.name}\\{path.name:<22} {rows[language]:>9,} rows")

GAPS = pd.DataFrame(UNTRANSLATED, columns=["chapter", "col_ar", "col_en", "val_ar",
                                           "val_en", "rows", "country", "indicator", "year"])
gap_count = len(GAPS.drop_duplicates(["chapter", "col_ar", "val_ar"]))
print(f"\nDistinct Arabic values with no dictionary entry: {gap_count}")
if gap_count:
    print("They are listed in each chapter's own need manual intervention_<Chapter>.txt,")
    print("with the country, indicator and year they appear in, and in GAPS below.")
    print("Ask Claude Code to fill the dictionary gaps, then run this notebook again.")

# ------------------------------------------------------- the shared log
records = list(FINDINGS) + gap_records(UNTRANSLATED)
for chapter, failures in FAILURES.items():
    for failure in failures:
        records.append({
            "kind": f"sheet could not be processed ({failure['step']})",
            "chapter": chapter, "file": failure["file"], "sheet": failure["sheet"],
            "detail": failure["error"],
        })

paths, count = save_inconsistencies("1. LONG FILES", records, chapters=chapters)
print(f"\n{count} inconsistency(ies) recorded across {len(paths)} file(s)")

## Run summary - failures grouped by chapter

In [ ]:
"""
CELL: Run summary - every failure from the run above, grouped by chapter.
"""
print("\n" + "=" * 70)
print("RUN SUMMARY - FAILURES BY CHAPTER")
print("=" * 70)

if not FAILURES:
    print("No failures. All files/sheets processed successfully.")
else:
    total = sum(len(v) for v in FAILURES.values())
    print(f"{total} failure(s) across {len(FAILURES)} chapter(s):\n")
    for chapter, failures in FAILURES.items():
        print(f"{chapter} ({len(failures)} failure(s)):")
        for f in failures:
            print(f"  - {f['file']} | Sheet={f['sheet']} | Step={f['step']} | {f['error']}")
        print()


## Dictionary gaps

Every Arabic value the dictionary could not translate, one row per value - the
shape `update_dictionary()` takes. Fill in `val_en` by position, never by
retyping the Arabic key, then call `update_dictionary(filled, chapter="<Chapter>")`
and re-run this notebook.

In [ ]:
"""
CELL: The values with no English translation, as a table to read here.
"""
GAPS.drop_duplicates(["chapter", "col_ar", "val_ar"])